# Module 7 • Hugging Face and Pretrained Transformer Workflows

# Lesson 41 • Machine Translation with Pretrained Multilingual Transformers

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Intermediate  
**Estimated study time:** 190–230 minutes  
**Execution target:** CPU by default

---

## Scope

This lesson develops machine translation workflows with pretrained multilingual
Transformer models. It covers translation task formulation, source and target
language control, multilingual tokenization, encoder–decoder inference, decoding,
evaluation, confidence intervals, error analysis, checkpoint metadata, and
responsible model selection.

The notebook contains:

1. a complete offline CPU translation experiment using a compact Transformer;
2. optional Hugging Face workflows for MarianMT, NLLB-200, M2M-100, and mT5-style
   sequence-to-sequence models, disabled by default.

## Learning Objectives

After completing this lesson, the learner should be able to:

- distinguish bilingual and multilingual translation models;
- explain source-language and target-language control;
- prepare parallel corpora and deterministic splits;
- build separate source and target vocabularies;
- train a compact Transformer translation model;
- perform greedy translation;
- explain beam search and length normalization;
- calculate exact match, token accuracy, BLEU-like, and chrF-like metrics;
- estimate confidence intervals with bootstrap resampling;
- analyze lexical, word-order, morphology, agreement, and length errors;
- structure Hugging Face MarianMT, NLLB-200, M2M-100, and mT5 workflows;
- assess Arabic morphology, clitics, script, dialect, and tashkeel requirements.

## Table of Contents

1. Machine Translation Tasks
2. Bilingual Versus Multilingual Models
3. Encoder–Decoder Translation
4. Language Control
5. Parallel Corpora
6. Data Leakage and Splits
7. Offline Translation Corpus
8. Train, Validation, and Test Splits
9. Tokenization
10. Source and Target Vocabularies
11. Numerical Encoding
12. Dataset and Dynamic Padding
13. Positional Encoding
14. Transformer Translation Model
15. Shape and Mask Inspection
16. Padding-Aware Sequence Loss
17. Training Utilities
18. Model Training
19. Learning Curves
20. Greedy Translation
21. Qualitative Translation
22. Exact Match
23. Token Accuracy
24. BLEU-Like Evaluation
25. chrF-Like Evaluation
26. Test Evaluation
27. Bootstrap Confidence Intervals
28. Error Taxonomy
29. Lexical Errors
30. Word-Order Errors
31. Agreement and Morphology Errors
32. Length Errors
33. Beam Search Foundations
34. Length Normalization
35. Checkpoint Saving and Reloading
36. Optional Hugging Face Setup
37. Optional MarianMT Workflow
38. Optional NLLB-200 Workflow
39. Optional M2M-100 Workflow
40. Optional mT5 Workflow
41. Evaluation Beyond BLEU
42. Arabic and Multilingual Considerations
43. Reproducibility and Reporting
44. Knowledge Check
45. Exercises
46. Summary and Next Lesson

# 1. Machine Translation Tasks

Machine translation maps a source-language sequence to a target-language sequence.

In [ ]:
import copy
import importlib.util
import math
import platform
import random
import re
import tempfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader, Dataset

mt_tasks = pd.DataFrame(
    [
        ('Bilingual', 'one source-target pair'),
        ('Multilingual', 'many language pairs'),
        ('Many-to-one', 'multiple sources to one target'),
        ('One-to-many', 'one source to multiple targets'),
        ('Zero-shot', 'unseen language direction'),
    ],
    columns=['Setting', 'Description'],
)

mt_tasks

# 2. Bilingual Versus Multilingual Models

Bilingual models specialize in one direction. Multilingual models share
parameters across many languages and may support transfer between low-resource
and high-resource pairs.

In [ ]:
model_families = pd.DataFrame(
    [
        ('MarianMT', 'often bilingual or small multilingual groups'),
        ('NLLB-200', 'large-scale multilingual translation'),
        ('M2M-100', 'direct many-to-many translation'),
        ('mT5', 'multilingual text-to-text model'),
    ],
    columns=['Family', 'Typical use'],
)

model_families

# 3. Encoder–Decoder Translation

The encoder represents the source sentence. The decoder generates the target
sentence autoregressively while attending to encoder memory.

# 4. Language Control

Multilingual models may require:

- a source-language code;
- a forced target-language token;
- a task prefix;
- model-specific tokenizer settings.

In [ ]:
language_control = pd.DataFrame(
    [
        ('MarianMT', 'checkpoint usually defines direction'),
        ('NLLB-200', 'src_lang plus forced BOS target token'),
        ('M2M-100', 'src_lang plus target language token'),
        ('mT5', 'text-to-text task prefix or fine-tuned convention'),
    ],
    columns=['Model family', 'Language control'],
)

language_control

# 5. Parallel Corpora

A parallel corpus contains aligned source and target sentences. Quality concerns
include duplicate pairs, misalignment, contamination, inconsistent normalization,
and train-test overlap.

# 6. Data Leakage and Splits

Near-duplicate source or target sentences should not cross data splits. For
benchmark comparisons, use the official split whenever available.

# 7. Offline Translation Corpus

The executable experiment uses a controlled English-to-French command corpus.
It contains lexical choice, adjective position, and gender agreement.

In [ ]:
verbs = {
    'open': 'ouvre',
    'close': 'ferme',
    'find': 'trouve',
    'take': 'prends',
    'move': 'deplace',
    'inspect': 'inspecte',
}

nouns = {
    'door': ('la', 'porte', 'f'),
    'window': ('la', 'fenetre', 'f'),
    'box': ('la', 'boite', 'f'),
    'book': ('le', 'livre', 'm'),
    'key': ('la', 'cle', 'f'),
    'report': ('le', 'rapport', 'm'),
}

modifiers = {
    'red': ('after', 'rouge', 'rouge'),
    'blue': ('after', 'bleu', 'bleue'),
    'green': ('after', 'vert', 'verte'),
    'yellow': ('after', 'jaune', 'jaune'),
    'small': ('before', 'petit', 'petite'),
    'big': ('before', 'grand', 'grande'),
}

def build_translation(verb: str, noun: str, modifier: str) -> tuple[str, str]:
    article, noun_fr, gender = nouns[noun]
    position, masculine, feminine = modifiers[modifier]
    adjective = feminine if gender == 'f' else masculine
    source = f'{verb} the {modifier} {noun}'
    if position == 'before':
        target = f'{verbs[verb]} {article} {adjective} {noun_fr}'
    else:
        target = f'{verbs[verb]} {article} {noun_fr} {adjective}'
    return source, target

rows = []
for verb in verbs:
    for noun in nouns:
        for modifier in modifiers:
            source, target = build_translation(verb, noun, modifier)
            rows.append({'source': source, 'target': target, 'verb': verb, 'noun': noun, 'modifier': modifier})

dataset = pd.DataFrame(rows)
print('Sentence pairs:', len(dataset))
dataset.sample(8, random_state=42)[['source', 'target']].reset_index(drop=True)

# 8. Train, Validation, and Test Splits

In [ ]:
train_frame, test_frame = train_test_split(
    dataset,
    test_size=0.15,
    random_state=42,
)

train_frame, validation_frame = train_test_split(
    train_frame,
    test_size=0.1765,
    random_state=42,
)

train_frame = train_frame.reset_index(drop=True)
validation_frame = validation_frame.reset_index(drop=True)
test_frame = test_frame.reset_index(drop=True)

pd.Series({
    'training': len(train_frame),
    'validation': len(validation_frame),
    'test': len(test_frame),
})

# 9. Tokenization

In [ ]:
TOKEN_PATTERN = re.compile(r"\b\w+(?:[-']\w+)*\b", flags=re.UNICODE)

def tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(text.lower())

tokenize('Open the small blue window.')

# 10. Source and Target Vocabularies

In [ ]:
PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'
BOS_TOKEN = '<BOS>'
EOS_TOKEN = '<EOS>'

class Vocabulary:
    def __init__(self, texts, include_bos: bool):
        counts = Counter(token for text in texts for token in tokenize(text))
        specials = [PAD_TOKEN, UNK_TOKEN]
        if include_bos:
            specials.append(BOS_TOKEN)
        specials.append(EOS_TOKEN)
        self.index_to_token = specials + sorted(counts)
        self.token_to_index = {token: index for index, token in enumerate(self.index_to_token)}
        self.pad_id = self.token_to_index[PAD_TOKEN]
        self.unk_id = self.token_to_index[UNK_TOKEN]
        self.eos_id = self.token_to_index[EOS_TOKEN]
        self.bos_id = self.token_to_index[BOS_TOKEN] if include_bos else None

    def __len__(self):
        return len(self.index_to_token)

    def encode(self, text: str, add_bos: bool = False) -> list[int]:
        ids = []
        if add_bos:
            ids.append(self.bos_id)
        ids.extend(self.token_to_index.get(token, self.unk_id) for token in tokenize(text))
        ids.append(self.eos_id)
        return ids

    def decode(self, ids) -> list[str]:
        output = []
        for token_id in ids:
            token = self.index_to_token[int(token_id)]
            if token == EOS_TOKEN:
                break
            if token not in {PAD_TOKEN, BOS_TOKEN}:
                output.append(token)
        return output

source_vocabulary = Vocabulary(train_frame['source'], include_bos=False)
target_vocabulary = Vocabulary(train_frame['target'], include_bos=True)

print('Source vocabulary:', len(source_vocabulary))
print('Target vocabulary:', len(target_vocabulary))

# 11. Numerical Encoding

In [ ]:
print(source_vocabulary.encode(train_frame.loc[0, 'source']))
print(target_vocabulary.encode(train_frame.loc[0, 'target'], add_bos=True))

# 12. Dataset and Dynamic Padding

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        return {
            'source_ids': torch.tensor(source_vocabulary.encode(row['source']), dtype=torch.long),
            'target_ids': torch.tensor(target_vocabulary.encode(row['target'], add_bos=True), dtype=torch.long),
            'source_text': row['source'],
            'target_text': row['target'],
        }

def collate_batch(batch):
    max_source = max(len(item['source_ids']) for item in batch)
    max_target = max(len(item['target_ids']) for item in batch)
    source_ids = torch.full((len(batch), max_source), source_vocabulary.pad_id, dtype=torch.long)
    target_ids = torch.full((len(batch), max_target), target_vocabulary.pad_id, dtype=torch.long)
    for row, item in enumerate(batch):
        source_ids[row, :len(item['source_ids'])] = item['source_ids']
        target_ids[row, :len(item['target_ids'])] = item['target_ids']
    return {
        'source_ids': source_ids,
        'target_ids': target_ids,
        'source_padding_mask': source_ids == source_vocabulary.pad_id,
        'source_texts': [item['source_text'] for item in batch],
        'target_texts': [item['target_text'] for item in batch],
    }

train_loader = DataLoader(TranslationDataset(train_frame), batch_size=16, shuffle=True, collate_fn=collate_batch, generator=torch.Generator().manual_seed(42))
validation_loader = DataLoader(TranslationDataset(validation_frame), batch_size=16, shuffle=False, collate_fn=collate_batch)
test_loader = DataLoader(TranslationDataset(test_frame), batch_size=16, shuffle=False, collate_fn=collate_batch)

sample_batch = next(iter(train_loader))
print(sample_batch['source_ids'].shape, sample_batch['target_ids'].shape)

# 13. Positional Encoding

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, model_dimension: int, maximum_length: int = 128):
        super().__init__()
        encoding = torch.zeros(maximum_length, model_dimension)
        positions = torch.arange(maximum_length, dtype=torch.float32).unsqueeze(1)
        rates = torch.exp(torch.arange(0, model_dimension, 2, dtype=torch.float32) * (-math.log(10000.0) / model_dimension))
        encoding[:, 0::2] = torch.sin(positions * rates)
        encoding[:, 1::2] = torch.cos(positions * rates)
        self.register_buffer('encoding', encoding.unsqueeze(0))

    def forward(self, embeddings: torch.Tensor) -> torch.Tensor:
        return embeddings + self.encoding[:, :embeddings.size(1), :]

# 14. Transformer Translation Model

In [ ]:
class TransformerTranslator(nn.Module):
    def __init__(self, source_size: int, target_size: int, model_dimension: int = 48, heads: int = 4, encoder_layers: int = 2, decoder_layers: int = 2, feed_forward: int = 96, dropout: float = 0.10):
        super().__init__()
        self.model_dimension = model_dimension
        self.source_embedding = nn.Embedding(source_size, model_dimension, padding_idx=source_vocabulary.pad_id)
        self.target_embedding = nn.Embedding(target_size, model_dimension, padding_idx=target_vocabulary.pad_id)
        self.position = PositionalEncoding(model_dimension)
        self.transformer = nn.Transformer(
            d_model=model_dimension,
            nhead=heads,
            num_encoder_layers=encoder_layers,
            num_decoder_layers=decoder_layers,
            dim_feedforward=feed_forward,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.output_layer = nn.Linear(model_dimension, target_size)

    def causal_mask(self, length: int, device: torch.device) -> torch.Tensor:
        return torch.triu(torch.ones(length, length, dtype=torch.bool, device=device), diagonal=1)

    def encode(self, source_ids, source_padding_mask):
        embeddings = self.source_embedding(source_ids) * math.sqrt(self.model_dimension)
        return self.transformer.encoder(self.position(embeddings), src_key_padding_mask=source_padding_mask)

    def decode(self, target_ids, memory, target_padding_mask, source_padding_mask):
        embeddings = self.target_embedding(target_ids) * math.sqrt(self.model_dimension)
        return self.transformer.decoder(
            self.position(embeddings),
            memory,
            tgt_mask=self.causal_mask(target_ids.size(1), target_ids.device),
            tgt_key_padding_mask=target_padding_mask,
            memory_key_padding_mask=source_padding_mask,
        )

    def forward(self, source_ids, target_input_ids, source_padding_mask, target_padding_mask):
        memory = self.encode(source_ids, source_padding_mask)
        decoded = self.decode(target_input_ids, memory, target_padding_mask, source_padding_mask)
        return {'logits': self.output_layer(decoded), 'memory': memory, 'decoded': decoded}

DEVICE = torch.device('cpu')
torch.manual_seed(42)
model = TransformerTranslator(len(source_vocabulary), len(target_vocabulary)).to(DEVICE)
print('Trainable parameters:', sum(parameter.numel() for parameter in model.parameters()))

# 15. Shape and Mask Inspection

In [ ]:
source_ids = sample_batch['source_ids'].to(DEVICE)
target_ids = sample_batch['target_ids'].to(DEVICE)
decoder_inputs = target_ids[:, :-1]
with torch.no_grad():
    output = model(
        source_ids,
        decoder_inputs,
        sample_batch['source_padding_mask'].to(DEVICE),
        decoder_inputs == target_vocabulary.pad_id,
    )
print('Memory:', output['memory'].shape)
print('Logits:', output['logits'].shape)
pd.DataFrame(model.causal_mask(decoder_inputs.size(1), DEVICE).int().cpu().numpy())

# 16. Padding-Aware Sequence Loss

In [ ]:
loss_function = nn.CrossEntropyLoss(ignore_index=target_vocabulary.pad_id)

def sequence_loss(logits: torch.Tensor, expected_ids: torch.Tensor) -> torch.Tensor:
    return loss_function(logits.reshape(-1, logits.size(-1)), expected_ids.reshape(-1))

# 17. Training Utilities

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def evaluate_loss(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    losses = []
    with torch.no_grad():
        for batch in loader:
            source_ids = batch['source_ids'].to(DEVICE)
            target_ids = batch['target_ids'].to(DEVICE)
            decoder_inputs = target_ids[:, :-1]
            expected_ids = target_ids[:, 1:]
            output = model(
                source_ids,
                decoder_inputs,
                batch['source_padding_mask'].to(DEVICE),
                decoder_inputs == target_vocabulary.pad_id,
            )
            losses.append(float(sequence_loss(output['logits'], expected_ids).item()))
    return float(np.mean(losses))

# 18. Model Training

In [ ]:
def train_model(model: nn.Module, epochs: int = 50, learning_rate: float = 0.003, patience: int = 9):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    best_state = copy.deepcopy(model.state_dict())
    best_validation_loss = float('inf')
    without_improvement = 0
    history = []

    for epoch in range(epochs):
        model.train()
        train_losses = []
        grad_norms = []
        for batch in train_loader:
            source_ids = batch['source_ids'].to(DEVICE)
            target_ids = batch['target_ids'].to(DEVICE)
            decoder_inputs = target_ids[:, :-1]
            expected_ids = target_ids[:, 1:]
            optimizer.zero_grad()
            output = model(
                source_ids,
                decoder_inputs,
                batch['source_padding_mask'].to(DEVICE),
                decoder_inputs == target_vocabulary.pad_id,
            )
            loss = sequence_loss(output['logits'], expected_ids)
            loss.backward()
            grad_norm = clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            train_losses.append(float(loss.item()))
            grad_norms.append(float(grad_norm))

        validation_loss = evaluate_loss(model, validation_loader)
        history.append({
            'epoch': epoch,
            'training_loss': float(np.mean(train_losses)),
            'validation_loss': validation_loss,
            'validation_perplexity': math.exp(min(validation_loss, 20.0)),
            'gradient_norm': float(np.mean(grad_norms)),
        })

        if validation_loss < best_validation_loss - 1e-5:
            best_validation_loss = validation_loss
            best_state = copy.deepcopy(model.state_dict())
            without_improvement = 0
        else:
            without_improvement += 1

        if without_improvement >= patience:
            break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

set_seed(42)
trained_model, training_history = train_model(model)
print('Epochs completed:', len(training_history))
print('Best validation loss:', round(training_history['validation_loss'].min(), 4))

# 19. Learning Curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(training_history['epoch'], training_history['training_loss'], label='Training loss')
plt.plot(training_history['epoch'], training_history['validation_loss'], label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Translation Learning Curves')
plt.legend()
plt.tight_layout()
plt.show()

# 20. Greedy Translation

In [ ]:
def greedy_translate(model: TransformerTranslator, source_text: str, maximum_length: int = 10) -> dict:
    model.eval()
    source_ids = torch.tensor([source_vocabulary.encode(source_text)], dtype=torch.long, device=DEVICE)
    source_padding_mask = source_ids == source_vocabulary.pad_id
    with torch.no_grad():
        memory = model.encode(source_ids, source_padding_mask)
        generated = [target_vocabulary.bos_id]
        probabilities = []
        for _ in range(maximum_length):
            target_input = torch.tensor([generated], dtype=torch.long, device=DEVICE)
            decoded = model.decode(
                target_input,
                memory,
                target_input == target_vocabulary.pad_id,
                source_padding_mask,
            )
            logits = model.output_layer(decoded[:, -1, :])
            probs = torch.softmax(logits, dim=1)
            next_id = int(probs.argmax(dim=1).item())
            generated.append(next_id)
            probabilities.append(float(probs[0, next_id].item()))
            if next_id == target_vocabulary.eos_id:
                break
    return {
        'translation': ' '.join(target_vocabulary.decode(generated)),
        'generated_ids': generated,
        'probabilities': probabilities,
    }

greedy_translate(trained_model, 'open the red door')

# 21. Qualitative Translation

In [ ]:
qualitative = []
for row in test_frame.head(10).itertuples(index=False):
    result = greedy_translate(trained_model, row.source)
    qualitative.append({
        'source': row.source,
        'reference': row.target,
        'prediction': result['translation'],
        'mean_confidence': float(np.mean(result['probabilities'])),
    })
pd.DataFrame(qualitative)

# 22. Exact Match

In [ ]:
def normalize_text(text: str) -> str:
    return ' '.join(tokenize(text))

def exact_match(reference: str, prediction: str) -> float:
    return float(normalize_text(reference) == normalize_text(prediction))

# 23. Token Accuracy

In [ ]:
def positional_token_accuracy(reference: str, prediction: str) -> float:
    reference_tokens = tokenize(reference)
    prediction_tokens = tokenize(prediction)
    denominator = max(len(reference_tokens), len(prediction_tokens), 1)
    matches = sum(left == right for left, right in zip(reference_tokens, prediction_tokens))
    return matches / denominator

# 24. BLEU-Like Evaluation

In [ ]:
def extract_ngrams(tokens: list[str], order: int) -> Counter:
    return Counter(tuple(tokens[index:index + order]) for index in range(len(tokens) - order + 1))

def sentence_bleu_like(reference: str, prediction: str, maximum_order: int = 4) -> float:
    reference_tokens = tokenize(reference)
    prediction_tokens = tokenize(prediction)
    if not prediction_tokens:
        return 0.0
    precisions = []
    for order in range(1, maximum_order + 1):
        predicted = extract_ngrams(prediction_tokens, order)
        reference_ngrams = extract_ngrams(reference_tokens, order)
        clipped = sum(min(count, reference_ngrams[ngram]) for ngram, count in predicted.items())
        total = sum(predicted.values())
        precisions.append((clipped + 1.0) / (total + 1.0))
    ref_len = len(reference_tokens)
    pred_len = len(prediction_tokens)
    brevity_penalty = 1.0 if pred_len > ref_len else math.exp(1.0 - ref_len / max(pred_len, 1))
    geometric_mean = math.exp(sum(math.log(max(value, 1e-12)) for value in precisions) / maximum_order)
    return brevity_penalty * geometric_mean

# 25. chrF-Like Evaluation

chrF compares character n-grams and is often informative for morphologically rich
languages.

In [ ]:
def character_ngrams(text: str, order: int) -> Counter:
    normalized = normalize_text(text).replace(' ', '')
    return Counter(normalized[index:index + order] for index in range(max(len(normalized) - order + 1, 0)))

def chrf_like(reference: str, prediction: str, maximum_order: int = 6, beta: float = 2.0) -> float:
    scores = []
    for order in range(1, maximum_order + 1):
        reference_ngrams = character_ngrams(reference, order)
        prediction_ngrams = character_ngrams(prediction, order)
        overlap = sum((reference_ngrams & prediction_ngrams).values())
        precision = overlap / max(sum(prediction_ngrams.values()), 1)
        recall = overlap / max(sum(reference_ngrams.values()), 1)
        beta_squared = beta ** 2
        denominator = beta_squared * precision + recall
        score = 0.0 if denominator == 0 else (1 + beta_squared) * precision * recall / denominator
        scores.append(score)
    return float(np.mean(scores))

# 26. Test Evaluation

In [ ]:
def evaluate_translations(model: TransformerTranslator, frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for row in frame.itertuples(index=False):
        result = greedy_translate(model, row.source)
        prediction = result['translation']
        rows.append({
            'source': row.source,
            'reference': row.target,
            'prediction': prediction,
            'exact_match': exact_match(row.target, prediction),
            'token_accuracy': positional_token_accuracy(row.target, prediction),
            'bleu_like': sentence_bleu_like(row.target, prediction),
            'chrf_like': chrf_like(row.target, prediction),
            'mean_confidence': float(np.mean(result['probabilities'])),
        })
    return pd.DataFrame(rows)

test_results = evaluate_translations(trained_model, test_frame)
test_results[['exact_match', 'token_accuracy', 'bleu_like', 'chrf_like']].mean()

# 27. Bootstrap Confidence Intervals

In [ ]:
def bootstrap_mean_interval(values, repetitions: int = 1000, seed: int = 42) -> dict:
    values = np.asarray(values, dtype=float)
    generator = np.random.default_rng(seed)
    means = []
    for _ in range(repetitions):
        indices = generator.integers(0, len(values), size=len(values))
        means.append(float(values[indices].mean()))
    return {
        'mean': float(values.mean()),
        'ci_lower': float(np.quantile(means, 0.025)),
        'ci_upper': float(np.quantile(means, 0.975)),
    }

pd.DataFrame([
    {'metric': 'Exact match', **bootstrap_mean_interval(test_results['exact_match'])},
    {'metric': 'BLEU-like', **bootstrap_mean_interval(test_results['bleu_like'])},
    {'metric': 'chrF-like', **bootstrap_mean_interval(test_results['chrf_like'])},
])

# 28. Error Taxonomy

In [ ]:
def categorize_translation_error(reference: str, prediction: str) -> str:
    ref_tokens = tokenize(reference)
    pred_tokens = tokenize(prediction)
    if ref_tokens == pred_tokens:
        return 'correct'
    if len(pred_tokens) < len(ref_tokens):
        return 'too short'
    if len(pred_tokens) > len(ref_tokens):
        return 'too long'
    if set(ref_tokens) == set(pred_tokens):
        return 'word order'
    shared = set(ref_tokens) & set(pred_tokens)
    if len(shared) >= max(len(set(ref_tokens)) - 1, 1):
        return 'agreement or morphology'
    return 'lexical error'

test_results['error_type'] = [
    categorize_translation_error(reference, prediction)
    for reference, prediction in zip(test_results['reference'], test_results['prediction'])
]
test_results['error_type'].value_counts()

# 29. Lexical Errors

Lexical errors select the wrong content word or omit a required translation.

# 30. Word-Order Errors

Word-order errors contain mostly correct words in an incorrect sequence.

# 31. Agreement and Morphology Errors

Agreement errors include incorrect adjective, gender, number, or case morphology.

# 32. Length Errors

In [ ]:
test_results['reference_length'] = test_results['reference'].map(lambda text: len(tokenize(text)))
test_results['prediction_length'] = test_results['prediction'].map(lambda text: len(tokenize(text)))
test_results[['reference_length', 'prediction_length']].describe()

# 33. Beam Search Foundations

Beam search retains several partial translations and expands them jointly.

In [ ]:
beam_state = pd.DataFrame(
    [
        ('token_ids', 'generated prefix'),
        ('log_probability', 'accumulated score'),
        ('finished', 'EOS status'),
        ('normalized_score', 'length-adjusted score'),
    ],
    columns=['Field', 'Purpose'],
)
beam_state

# 34. Length Normalization

In [ ]:
def normalized_score(log_probability: float, length: int, alpha: float = 0.7) -> float:
    return log_probability / max(length, 1) ** alpha

pd.DataFrame([
    (-2.0, 4, normalized_score(-2.0, 4)),
    (-2.6, 6, normalized_score(-2.6, 6)),
], columns=['Log probability', 'Length', 'Normalized score'])

# 35. Checkpoint Saving and Reloading

In [ ]:
with tempfile.TemporaryDirectory() as directory:
    checkpoint_path = Path(directory) / 'translator.pt'
    torch.save({
        'model_state_dict': trained_model.state_dict(),
        'source_vocabulary': source_vocabulary.index_to_token,
        'target_vocabulary': target_vocabulary.index_to_token,
    }, checkpoint_path)
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    reloaded_model = TransformerTranslator(
        source_size=len(checkpoint['source_vocabulary']),
        target_size=len(checkpoint['target_vocabulary']),
    ).to(DEVICE)
    reloaded_model.load_state_dict(checkpoint['model_state_dict'])
    reloaded_translation = greedy_translate(reloaded_model, test_frame.loc[0, 'source'])['translation']
print(reloaded_translation)

# 36. Optional Hugging Face Setup

In [ ]:
TRANSFORMERS_AVAILABLE = importlib.util.find_spec('transformers') is not None
SENTENCEPIECE_AVAILABLE = importlib.util.find_spec('sentencepiece') is not None
RUN_HUGGING_FACE_DEMOS = False
USE_LOCAL_FILES_ONLY = True

pd.Series({
    'transformers installed': TRANSFORMERS_AVAILABLE,
    'sentencepiece installed': SENTENCEPIECE_AVAILABLE,
    'run demos': RUN_HUGGING_FACE_DEMOS,
    'local files only': USE_LOCAL_FILES_ONLY,
})

# 37. Optional MarianMT Workflow

In [ ]:
if TRANSFORMERS_AVAILABLE and RUN_HUGGING_FACE_DEMOS:
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

    MARIAN_MODEL_ID = 'Helsinki-NLP/opus-mt-en-fr'
    tokenizer = AutoTokenizer.from_pretrained(MARIAN_MODEL_ID, local_files_only=USE_LOCAL_FILES_ONLY)
    marian_model = AutoModelForSeq2SeqLM.from_pretrained(MARIAN_MODEL_ID, local_files_only=USE_LOCAL_FILES_ONLY).to('cpu')
    inputs = tokenizer('Open the red door.', return_tensors='pt')
    with torch.no_grad():
        generated = marian_model.generate(**inputs, num_beams=4, max_new_tokens=32)
    print(tokenizer.decode(generated[0], skip_special_tokens=True))
else:
    print('Optional MarianMT workflow skipped.')

# 38. Optional NLLB-200 Workflow

In [ ]:
if TRANSFORMERS_AVAILABLE and RUN_HUGGING_FACE_DEMOS:
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

    NLLB_MODEL_ID = 'facebook/nllb-200-distilled-600M'
    nllb_tokenizer = AutoTokenizer.from_pretrained(
        NLLB_MODEL_ID,
        src_lang='eng_Latn',
        local_files_only=USE_LOCAL_FILES_ONLY,
    )
    nllb_model = AutoModelForSeq2SeqLM.from_pretrained(
        NLLB_MODEL_ID,
        local_files_only=USE_LOCAL_FILES_ONLY,
    ).to('cpu')
    encoded = nllb_tokenizer('Open the red door.', return_tensors='pt')
    forced_bos = nllb_tokenizer.convert_tokens_to_ids('fra_Latn')
    with torch.no_grad():
        generated = nllb_model.generate(**encoded, forced_bos_token_id=forced_bos, max_new_tokens=32)
    print(nllb_tokenizer.decode(generated[0], skip_special_tokens=True))
else:
    print('Optional NLLB-200 workflow skipped.')

# 39. Optional M2M-100 Workflow

In [ ]:
if TRANSFORMERS_AVAILABLE and RUN_HUGGING_FACE_DEMOS:
    from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

    M2M_MODEL_ID = 'facebook/m2m100_418M'
    m2m_tokenizer = M2M100Tokenizer.from_pretrained(M2M_MODEL_ID, local_files_only=USE_LOCAL_FILES_ONLY)
    m2m_tokenizer.src_lang = 'en'
    m2m_model = M2M100ForConditionalGeneration.from_pretrained(M2M_MODEL_ID, local_files_only=USE_LOCAL_FILES_ONLY).to('cpu')
    encoded = m2m_tokenizer('Open the red door.', return_tensors='pt')
    with torch.no_grad():
        generated = m2m_model.generate(**encoded, forced_bos_token_id=m2m_tokenizer.get_lang_id('fr'), max_new_tokens=32)
    print(m2m_tokenizer.decode(generated[0], skip_special_tokens=True))
else:
    print('Optional M2M-100 workflow skipped.')

# 40. Optional mT5 Workflow

mT5 is a multilingual text-to-text model. Translation generally requires a
checkpoint fine-tuned for the desired translation task or an explicit task prefix
consistent with its fine-tuning data.

In [ ]:
if TRANSFORMERS_AVAILABLE and RUN_HUGGING_FACE_DEMOS:
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

    MT5_MODEL_ID = 'google/mt5-small'
    mt5_tokenizer = AutoTokenizer.from_pretrained(MT5_MODEL_ID, local_files_only=USE_LOCAL_FILES_ONLY)
    mt5_model = AutoModelForSeq2SeqLM.from_pretrained(MT5_MODEL_ID, local_files_only=USE_LOCAL_FILES_ONLY).to('cpu')
    prompt = 'translate English to French: Open the red door.'
    encoded = mt5_tokenizer(prompt, return_tensors='pt')
    with torch.no_grad():
        generated = mt5_model.generate(**encoded, max_new_tokens=32)
    print(mt5_tokenizer.decode(generated[0], skip_special_tokens=True))
else:
    print('Optional mT5 workflow skipped.')

# 41. Evaluation Beyond BLEU

Translation evaluation may include:

- chrF;
- COMET;
- BERTScore;
- adequacy and fluency judgments;
- terminology accuracy;
- morphology and agreement analysis;
- confidence intervals and paired significance tests.

In [ ]:
metric_summary = pd.DataFrame(
    [
        ('BLEU', 'word or subword n-gram precision'),
        ('chrF', 'character n-gram precision and recall'),
        ('COMET', 'learned quality estimation with references'),
        ('BERTScore', 'contextual embedding similarity'),
        ('Human evaluation', 'adequacy, fluency, terminology, faithfulness'),
    ],
    columns=['Metric', 'Focus'],
)
metric_summary

# 42. Arabic and Multilingual Considerations

Arabic translation is affected by:

- attached clitics;
- rich morphology;
- gender and number agreement;
- word-order variation;
- optional tashkeel;
- MSA and dialect differences;
- script normalization;
- named-entity transliteration.

In [ ]:
arabic_examples = pd.DataFrame(
    [
        ('وَسَيَكْتُبُونَهَا', 'وَ + سَ + يَكْتُبُونَ + هَا'),
        ('بِالْمَدْرَسَةِ', 'بِ + الْمَدْرَسَةِ'),
        ('كِتَابُهُمَا', 'كِتَابُ + هُمَا'),
    ],
    columns=['Fully vocalized form', 'Illustrative segmentation'],
)
arabic_examples

For fully vocalized Arabic translation tasks, tashkeel must remain in training,
validation, test, tokenization, decoding, and evaluation whenever it is part of the
task definition.

In [ ]:
arabic_mt_checks = pd.DataFrame(
    [
        ('Tashkeel', 'preserve and evaluate consistently'),
        ('Clitics', 'inspect segmentation and attachment'),
        ('Agreement', 'analyze gender and number morphology'),
        ('Variety', 'separate MSA and dialect evaluation'),
        ('Entities', 'inspect translation versus transliteration'),
    ],
    columns=['Check', 'Action'],
)
arabic_mt_checks

# 43. Reproducibility and Reporting

Report:

- corpus and split;
- source and target languages;
- tokenizer and revision;
- model and checkpoint revision;
- language-control tokens;
- maximum source and target lengths;
- decoding settings;
- BLEU, chrF, COMET, and human evaluation;
- confidence intervals and statistical tests;
- random seeds;
- software versions;
- hardware;
- limitations.

In [ ]:
reproducibility_metadata = pd.Series(
    {
        'training examples': len(train_frame),
        'validation examples': len(validation_frame),
        'test examples': len(test_frame),
        'source vocabulary': len(source_vocabulary),
        'target vocabulary': len(target_vocabulary),
        'device': str(DEVICE),
        'seed': 42,
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers installed': TRANSFORMERS_AVAILABLE,
        'optional demos enabled': RUN_HUGGING_FACE_DEMOS,
    },
    name='Lesson 41 experiment',
)
reproducibility_metadata

# 44. Knowledge Check

1. How do bilingual and multilingual MT models differ?
2. Why do multilingual models need language control?
3. What is a parallel corpus?
4. Why must near-duplicate pairs be separated across splits?
5. Why are decoder inputs shifted?
6. What does greedy decoding do?
7. Why can beam search improve translation?
8. What does BLEU-like evaluation emphasize?
9. Why is chrF useful for morphologically rich languages?
10. What does a bootstrap confidence interval estimate?
11. What is a lexical error?
12. What is a word-order error?
13. How do NLLB and M2M-100 specify target languages?
14. Why is a base mT5 checkpoint not automatically a translation system?
15. Why does tashkeel policy matter for Arabic translation?

# 45. Exercises

## Exercise 1 — New Language Pair

Replace the controlled corpus with another documented language pair.

## Exercise 2 — Beam Search

Implement beam search with length normalization.

## Exercise 3 — Established Metrics

Add SacreBLEU and chrF through established libraries.

## Exercise 4 — COMET

Add COMET evaluation in a compatible Python environment.

## Exercise 5 — Statistical Testing

Compare two systems with paired bootstrap resampling.

## Exercise 6 — MarianMT

Run a bilingual MarianMT checkpoint.

## Exercise 7 — NLLB-200

Translate one source sentence into two target languages.

## Exercise 8 — M2M-100

Compare direct and pivot translation.

## Exercise 9 — Arabic Translation

Build a fully vocalized Arabic↔English translation task.

## Exercise 10 — Error Analysis

Create a manual taxonomy for morphology, agreement, word order, and terminology.

## Challenge Exercises

1. Fine-tune MarianMT on a domain-specific corpus.
2. Compare MarianMT, NLLB-200, M2M-100, and mT5 under matched data.
3. Add confidence intervals for BLEU, chrF, COMET, and BERTScore.
4. Evaluate terminology consistency and named entities.
5. Publish a reproducible translation model card.

# 46. Summary and Next Lesson

In this lesson:

- bilingual and multilingual translation settings were distinguished;
- encoder–decoder translation and language-control mechanisms were explained;
- a controlled parallel corpus was prepared with deterministic splits;
- source and target vocabularies, padding, masks, and shifted targets were created;
- a complete CPU-only Transformer translation model was trained;
- greedy decoding generated translations;
- exact match, token accuracy, BLEU-like, and chrF-like metrics were calculated;
- bootstrap confidence intervals quantified evaluation uncertainty;
- lexical, word-order, agreement, morphology, and length errors were analyzed;
- optional MarianMT, NLLB-200, M2M-100, and mT5 workflows were provided;
- Arabic morphology, clitics, agreement, multilingual variation, and tashkeel were integrated.

## Next Lesson

**Lesson 42: Parameter-Efficient Fine-Tuning with PEFT and LoRA** introduces
adapters, low-rank updates, trainable-parameter accounting, memory trade-offs,
LoRA configuration, checkpoint merging, evaluation, and responsible deployment.

# References

- Hugging Face Transformers documentation: translation, generation, tokenizers,
  multilingual models, and sequence-to-sequence training.
- Tiedemann, J., & Thottingal, S. OPUS-MT and Marian.
- Fan, A. et al. M2M-100.
- Costa-jussà, M. R. et al. NLLB.
- Xue, L. et al. mT5.
- Papineni, K. et al. BLEU.
- Popović, M. chrF.
- Rei, R. et al. COMET.